# Phase 3 / D3 — Cross-Currency Basis / Dollar-Funding Carry

Stages 1–6 showed the 2007–2026 carry premium is an EM phenomenon and that every *standard* overlay
— crash hedges (St3), optimization (St4), momentum (St5), regime timing (St6) — fails to beat the
simple vol-targeted inverse-vol carry book net of costs; **D1** added that the option-implied crash
skew is likewise no edge. **D3** asks whether the **post-2008 cross-currency basis** — the CIP
deviation, i.e. the *dollar-funding premium* of Du–Tepper–Verdelhan (2018) — turns into carry alpha,
in **both** of its natural uses: a **time-series conditioner** (de-risk when dollar funding is
stressed; Brunnermeier–Nagel–Pedersen 2009; Avdjiev–Du–Koch–Shin 2019) **and** a **cross-sectional
signal** (basis-sorted / basis-adjusted / basis-orthogonalized carry).

**The bar (falsifiable).** Beat *both* the simple vol-targeted book (ALL net Sharpe **0.466**) *and*
the per-currency-RR-hedged book (**0.457**), net of costs, with Newey–West significance — or report a
null. Guardrails §6: no lookahead (signals sampled month-end, `ffill().shift(1)`, trailing windows
only), gross AND net (`forward_halfspreads`+`roundtrip_cost`), IR vs FXCTEM8.

**The battery** — matched **7-name** basis universe (`cip_basis` needs onshore fixings), tercile
inverse-vol, vol-targeted 10%:

| variant | signal (1M) | competing thesis |
|---|---|---|
| `basis` | `cip_basis` (long rich, short cheap) | DTV: high-rate ⇒ higher basis, so a basis sort ≈ carry — does it *add* anything? |
| `onshore` | `interest_diff_vs_usd` (= carry − basis) | the "pure rate" carry with the funding premium stripped |
| `tilthi`/`tiltlo` | `z(carry) ± z(basis)` | carry tilted **toward** rich-basis vs **toward** dollar-shortage (negative-basis) names |
| `clean` | carry ⟂ basis (`xs_residual`) | does the non-basis part of carry survive? |
| conditioner | `exposure_scalar(basis_stress_index)` on **ALL-27** | de-risk the whole book when dollar funding is stressed |

**Matched universe & window caveat.** `cip_basis` is gated by the onshore-fixing set, so the tradable
basis universe is just **7 EM names** (no G10) — an EM convertibility/NDF-basis study, not the G10
dollar-funding basis DTV studied. And the onshore fixings **end 2024-09-30**, so every basis-dependent
track is evaluated on the matched window **2007-05 → 2024-09**; the ALL-27 vanilla-carry track is
reconciled to the committed Stage-4 baseline over the full **2007-05 → 2026-06** window (§5). Comparator
for NW alpha = the **matched U7 vanilla carry**; the published 0.466 / 0.457 are the absolute bars.

In [1]:
# ===== 0. Setup =====
from pathlib import Path
import numpy as np, pandas as pd
import fx_utils as fx

OUT = Path('outputs')
g10_px = fx.load_wide('g10_fx_spot_forward')
em_px  = fx.load_wide('em_fx_spot_forward')
spots  = fx.spots_usd_per_fx(g10_px, em_px)
carry  = fx.carry_panel(g10_px, em_px, tenor='1M')
xret   = fx.excess_returns(spots, carry)
bmk    = fx.benchmark_returns()
rates  = fx.load_rates_panel()
hs_out, hs_pts = fx.forward_halfspreads('1M')
basis  = fx.cip_basis(g10_px, em_px, tenor='1M', rates=rates)   # forward-implied carry − onshore diff

BENCH      = 'FXCTEM8'
VOL_TARGET = 0.10
N_BUCKETS  = 3       # thin 7-name EM sort → terciles (~2 names/leg); quintiles would give 1/leg
N_ALL      = 5       # ALL-27 reconciliation book keeps the Stage-4 quintile convention
WFULL      = slice('2007-05-01', '2026-06-30')   # forwards/carry span → ALL-27 reconciliation
WD3        = slice('2007-05-01', '2024-09-30')   # onshore fixings end 2024-09 → basis-dependent tracks

# Matched universe: tradable 27 names ∩ cip_basis (onshore-fixing) coverage = 7 EM names.
UNIVERSE_ALL = [c for c in xret.columns if c not in ('HKD', 'DKK', 'CNY')]
U7 = [c for c in xret.columns if c in basis.columns and c not in ('HKD', 'DKK', 'CNY')]
BASIS_END = basis[U7].dropna(how='all').index.max()
print('xret', xret.shape, '| ALL', len(UNIVERSE_ALL), '| U7', len(U7), '| basis ends', BASIS_END.date())
print('U7:', U7)

xret (5094, 29) | ALL 27 | U7 7 | basis ends 2024-09-30
U7: ['CNH', 'HUF', 'ILS', 'INR', 'PLN', 'THB', 'TRY']


## 1. Signals and weight panels

The whole battery reuses the existing engine: `cip_basis = carry_panel − interest_diff_vs_usd` is an
identity, so the basis, the onshore-rate carry (= carry − basis), the tilts and the orthogonalized
("clean") carry are all compositions of library functions; only `basis_stress_index` (the aggregate
dollar-funding gauge for the conditioner) is new. Rate-derived panels live on the fixings calendar
(which overruns `xret`), so we `reindex` them to the return calendar — alignment only, no lookahead.

In [2]:
# ===== 1. Signals and weight panels =====
carryU = carry[U7]
basisU = basis[U7].reindex(xret.index)
onsh   = fx.interest_diff_vs_usd(tenor='1M', rates=rates)[U7].reindex(xret.index)  # ≡ carry − basis
zc, zb = fx.zscore_xs(carryU), fx.zscore_xs(basisU)
signals = {
    'carry':   carryU,                          # matched vanilla carry (baseline / NW comparator)
    'basis':   basisU,                          # (a) the basis component alone (long rich, short cheap)
    'onshore': onsh,                            # (b) pure onshore-rate carry (funding premium stripped)
    'tilthi':  zc + zb,                         # (c) carry tilted toward rich-basis names
    'tiltlo':  zc - zb,                         # (c) carry tilted toward dollar-shortage (neg-basis) names
    'clean':   fx.xs_residual(carryU, basisU),  # (d) carry orthogonalized to the basis
}
def vt(w): return fx.vol_target_weights(w, xret, target=VOL_TARGET)
w = {k: vt(fx.carry_portfolio(sig, xret, n_buckets=N_BUCKETS, universe=U7, weighting='inv_vol'))
     for k, sig in signals.items()}
# ALL-27 vanilla carry: reconciliation bridge to Stage 3/4 (net Sharpe ~0.466) + conditioner base.
w['all_carry'] = vt(fx.carry_portfolio(carry, xret, n_buckets=N_ALL,
                                       universe=UNIVERSE_ALL, weighting='inv_vol'))
# Conditioner: halve ALL-27 exposure when basis-derived dollar-funding stress is in its top quintile.
stress = fx.basis_stress_index(basisU, method='zmean')   # peaks Lehman-2008 / COVID-2020
scalar = fx.exposure_scalar(stress.reindex(xret.index), lookback=756, q=0.80, low_mult=0.5)
w['all_basiscond'] = w['all_carry'].mul(scalar.reindex(w['all_carry'].index, method='ffill').fillna(1.0), axis=0)
print('weight panels:', list(w))
print('conditioner de-risks', int((scalar.loc[WD3] < 1).sum()), 'of', int(scalar.loc[WD3].notna().sum()), 'WD3 days')

weight panels: ['carry', 'basis', 'onshore', 'tilthi', 'tiltlo', 'clean', 'all_carry', 'all_basiscond']
conditioner de-risks 761 of 4552 WD3 days


## 2. Tracks — gross and net of costs

`net = gross − roundtrip_cost` (new notional pays the outright half-spread; maintained notional rolls
at the points half-spread). Basis-dependent tracks are evaluated on the matched window **WD3**
(2007-05→2024-09); the ALL-27 reconciliation to Stage-4 over the full window is in §5.

In [3]:
# ===== 2. Gross / net tracks =====
LABEL = {'carry':'U7_carry', 'basis':'U7_basis', 'onshore':'U7_onshore', 'tilthi':'U7_tilthi',
         'tiltlo':'U7_tiltlo', 'clean':'U7_clean', 'all_carry':'ALL_carry', 'all_basiscond':'ALL_basiscond'}
tracks, costs = {}, {}
for k, wk in w.items():
    lbl = LABEL[k]
    g = fx.portfolio_returns(wk, xret).rename(f'{lbl}_gross')
    c = fx.roundtrip_cost(wk, hs_out, hs_pts)
    tracks[g.name] = g
    tracks[f'{lbl}_net'] = (g - c).rename(f'{lbl}_net')
    costs[lbl] = c
daily = pd.concat(list(tracks.values()) + [bmk[BENCH]], axis=1).loc[WD3]
print('daily panel (WD3)', daily.shape, '| tracks', len(tracks))

daily panel (WD3) (4552, 17) | tracks 16


## 3. Statistics — full metrics, IR, turnover, cost drag, NW alpha vs matched carry

IR vs FXCTEM8. NW (5-lag HAC) alpha of every net track on the **matched** `U7_carry_net`. All tracks
on WD3, so `ALL_carry_net` here is the **2007–2024 subsample** of the committed book (§5 reconciles the
full-window value to 0.4659).

In [4]:
# ===== 3. Statistics =====
strat_cols = [c for c in daily.columns if c != BENCH]
summary = pd.concat([
    fx.summary_stats(daily[strat_cols], benchmark=bmk[BENCH].loc[WD3]),
    fx.summary_stats(daily[[BENCH]]),
])
base = daily['U7_carry_net']                     # matched vanilla carry = NW comparator
meta = {}
for k, lbl in LABEL.items():
    to = fx.turnover(w[k].loc[WD3])
    for basis_lbl in ('gross', 'net'):
        col = f'{lbl}_{basis_lbl}'
        row = {'avg_monthly_turnover': to}
        if basis_lbl == 'net':
            n = daily[col]; c_win = costs[lbl].reindex(n.index)
            row['ann_cost_drag'] = float(c_win[n.notna()].mean() * fx.ANN_DAYS)
            if col != 'U7_carry_net':
                reg = fx.nw_regression(n, base.to_frame('carry'), lags=5)
                row['alpha_vs_carry_ann'] = reg['alpha_ann'] if reg else np.nan
                row['t_alpha_vs_carry']   = reg['alpha_t'] if reg else np.nan
        meta[col] = row
summary = summary.join(pd.DataFrame(meta).T)

# The conditioner's own test: NW alpha of the de-risked book over the un-conditioned ALL-27 book.
cond_reg = fx.nw_regression(daily['ALL_basiscond_net'], daily['ALL_carry_net'].to_frame('allcarry'), lags=5)

show = ['sharpe', 'info_ratio', 'max_drawdown', 'CVaR_99', 'skew',
        'avg_monthly_turnover', 'ann_cost_drag', 't_alpha_vs_carry']
print(f"conditioner vs un-conditioned ALL-27 (WD3): alpha {cond_reg['alpha_ann']:+.4f}/yr  t {cond_reg['alpha_t']:+.2f}")
summary[show].round(3)

conditioner vs un-conditioned ALL-27 (WD3): alpha +0.0036/yr  t +0.76


,sharpe,info_ratio,max_drawdown,CVaR_99,skew,avg_monthly_turnover,ann_cost_drag,t_alpha_vs_carry
U7_carry_gross,-0.163642,-0.215854,-0.626817,0.02967,-0.295046,0.446,NaN,NaN
U7_carry_net,-0.317246,-0.357128,-0.716549,0.029722,-0.291386,0.446,0.018,NaN
U7_basis_gross,0.15999,0.06825,-0.353863,0.029316,-0.102952,1.137,NaN,NaN
U7_basis_net,-0.125632,-0.14086,-0.535594,0.02959,-0.106062,1.137,0.034,-0.322
U7_onshore_gross,-0.295333,-0.354454,-0.693819,0.028545,-0.353318,0.234,NaN,NaN
U7_onshore_net,-0.40838,-0.463473,-0.744791,0.028567,-0.347961,0.234,0.013,-1.230
U7_tilthi_gross,0.059736,-0.005823,-0.517162,0.033007,-0.735963,0.952,NaN,NaN
U7_tilthi_net,-0.171261,-0.205843,-0.666208,0.033007,-0.727703,0.952,0.028,0.497
U7_tiltlo_gross,-0.492221,-0.573156,-0.751735,0.029621,-0.625283,0.768,NaN,NaN
U7_tiltlo_net,-0.687905,-0.77148,-0.82983,0.029781,-0.612964,0.768,0.023,-2.756


## 4. Basis-vs-carry spanning — the headline test

*Does the basis add anything carry doesn't already contain?* We test it **both ways** on U7 unit
gross-2/net-0 long/short factor books (gross returns, NW 5 lags): `BASIS~CARRY` (does the basis book's
alpha survive carry?) and `CARRY~BASIS` (does carry survive the basis?). `ONSHORE~CARRY` shows the
onshore-rate carry is a near-clone of the forward carry — the basis is only a small increment.

In [5]:
# ===== 4. Basis vs carry spanning =====
def factor(sig, name):
    wf = fx.carry_portfolio(sig, xret, n_buckets=N_BUCKETS, universe=U7, weighting='inv_vol')
    return fx.portfolio_returns(wf, xret, name).loc[WD3]
carry_f = factor(signals['carry'], 'CARRY')
basis_f = factor(signals['basis'], 'BASIS')
onsh_f  = factor(signals['onshore'], 'ONSHORE')

def span_row(y, X, xname):
    r = fx.nw_regression(y, X, lags=5)
    return {'alpha_ann': r['alpha_ann'], 'alpha_t': r['alpha_t'],
            'beta': r[f'beta_{xname}'], 't_beta': r[f't_{xname}'], 'r2': r['r2'], 'n': r['n']}
span = pd.DataFrame({
    'BASIS~CARRY':   span_row(basis_f, carry_f.to_frame('CARRY'), 'CARRY'),   # does basis survive carry?
    'CARRY~BASIS':   span_row(carry_f, basis_f.to_frame('BASIS'), 'BASIS'),   # does carry survive basis?
    'ONSHORE~CARRY': span_row(onsh_f, carry_f.to_frame('CARRY'), 'CARRY'),    # onshore ≈ forward carry
}).T[['alpha_ann', 'alpha_t', 'beta', 't_beta', 'r2', 'n']]
print('factor gross Sharpe: carry',
      round(float(fx.summary_stats(carry_f.to_frame('c')).loc['c', 'sharpe']), 3),
      '| basis', round(float(fx.summary_stats(basis_f.to_frame('b')).loc['b', 'sharpe']), 3),
      '| onshore', round(float(fx.summary_stats(onsh_f.to_frame('o')).loc['o', 'sharpe']), 3),
      '| corr(carry,basis)', round(carry_f.corr(basis_f), 3),
      '| corr(carry,onshore)', round(carry_f.corr(onsh_f), 3))
span.round(4)

factor gross Sharpe: carry -0.048 | basis 0.141 | onshore -0.093 | corr(carry,basis) 0.22 | corr(carry,onshore) 0.845


,alpha_ann,alpha_t,beta,t_beta,r2,n
BASIS~CARRY,0.0163,0.6589,0.2416,2.8404,0.0484,4545.0
CARRY~BASIS,-0.0077,-0.3432,0.2004,2.6732,0.0484,4545.0
ONSHORE~CARRY,-0.0055,-0.4507,0.8921,41.6429,0.7135,4545.0


## 5. Validation — universe, no-lookahead, reconciliation, window

In [6]:
# ===== 5. Validation =====
# (i) matched universe is exactly the 7 basis-covered tradable EM names
assert sorted(U7) == sorted(['HUF','PLN','TRY','ILS','THB','INR','CNH']), 'U7 mismatch'

# rebuild every rate/price input truncated at `cut` exactly as the production path (reindex to the
# return calendar BEFORE any rolling), so a match on an earlier inner window proves no leakage.
cut = '2015-06-30'
b_tr = fx.cip_basis(g10_px.loc[:cut], em_px.loc[:cut], tenor='1M', rates=rates.loc[:cut])[U7].reindex(xret.loc[:cut].index)
chk = slice('2012-01-01', '2015-05-31')
# (ii) no-lookahead: cip_basis on truncated data matches the full panel on the overlap tail
assert np.allclose(b_tr.iloc[-20:].values,
                   basisU.reindex(b_tr.index).iloc[-20:].values, equal_nan=True), 'lookahead in cip_basis'
# (iii-a) no-lookahead: conditioner scalar on truncated data matches the full run on the inner window
scal_tr = fx.exposure_scalar(fx.basis_stress_index(b_tr, method='zmean'), lookback=756, q=0.80, low_mult=0.5)
assert np.allclose(scalar.loc[chk].values, scal_tr.reindex(scalar.loc[chk].index).values, atol=1e-10), 'lookahead in conditioner scalar'
# (iii-b) no-lookahead: basis-sorted weight panel on truncated data matches the full run on the inner window
bw_full = fx.carry_portfolio(basisU, xret, n_buckets=N_BUCKETS, universe=U7, weighting='inv_vol')
bw_tr = fx.carry_portfolio(b_tr, xret.loc[:cut], n_buckets=N_BUCKETS, universe=U7, weighting='inv_vol')
assert np.allclose(bw_full.loc[chk].fillna(0).values,
                   bw_tr.loc[chk].reindex(columns=bw_full.columns).fillna(0).values, atol=1e-10), 'lookahead in basis weights'

# (iv) reconcile ALL-27 inv_vol-net Sharpe to the committed Stage-4 number (~0.466) over the FULL window
recon = (fx.portfolio_returns(w['all_carry'], xret)
         - fx.roundtrip_cost(w['all_carry'], hs_out, hs_pts)).rename('ALL_carry_net').loc[WFULL]
a = float(fx.summary_stats(recon.to_frame()).loc['ALL_carry_net', 'sharpe'])
s4 = pd.read_csv(OUT / 'stage4_weighting_comparison.csv', index_col=0)
b = float(s4.loc['ALL_inv_vol_net', 'sharpe'])
print(f'ALL-27 inv_vol-net Sharpe (full window) {a:.4f} vs Stage-4 {b:.4f} (d={a - b:+.5f})')
assert abs(a - b) < 1e-3, 'ALL-27 carry path drifted from Stage 4'

# (v) window end (basis tracks end 2024-09; ALL_carry subsample too), loose start, NW-alpha completeness
strat = [i for i in summary.index if i.startswith(('U7_', 'ALL_'))]
assert all(str(summary.loc[i, 'end']) == '2024-09-30' for i in strat), 'window end'
assert all(str(summary.loc[i, 'start']) <= '2008-01-31' for i in strat), 'window start'
non_base = [i for i in strat if i.endswith('_net') and i != 'U7_carry_net']
assert summary.loc[non_base, 't_alpha_vs_carry'].notna().all(), 'missing NW alpha'
print('checks passed')

ALL-27 inv_vol-net Sharpe (full window) 0.4659 vs Stage-4 0.4659 (d=+0.00000)
checks passed


## 6. Output + robustness

In [7]:
# ===== 6. Output =====
OUT.mkdir(exist_ok=True)
summary.to_csv(OUT / 'basis_carry_comparison.csv')
span.to_csv(OUT / 'basis_carry_spanning.csv')
net_cols = [c for c in daily.columns if c.endswith('_net')]
corr = daily[net_cols].corr()
corr.to_csv(OUT / 'basis_track_correlation.csv')
assert (OUT / 'basis_carry_comparison.csv').exists() and (OUT / 'basis_carry_spanning.csv').exists()
print('wrote: basis_carry_comparison.csv, basis_carry_spanning.csv, basis_track_correlation.csv')

# --- robustness: is the null a tenor / single-name / stress-method artifact? ---
def sharpe_net(sig_panel, uni, nb=3):
    wk = vt(fx.carry_portfolio(sig_panel, xret, n_buckets=nb, universe=uni, weighting='inv_vol'))
    n = (fx.portfolio_returns(wk, xret) - fx.roundtrip_cost(wk, hs_out, hs_pts)).rename('x').loc[WD3]
    return round(float(fx.summary_stats(n.to_frame()).loc['x', 'sharpe']), 3)

basis3 = fx.cip_basis(g10_px, em_px, tenor='3M', rates=rates)
U7b = [c for c in U7 if c in basis3.columns]; b3 = basis3[U7b].reindex(xret.index)
print('3M tenor : U7 carry', sharpe_net(carry[U7b], U7b), '| 3M basis-sort', sharpe_net(b3, U7b))
for drop in (['CNH'], ['TRY']):                 # 6 names → terciles still give 2/leg
    u = [c for c in U7 if c not in drop]
    print(f'drop {drop}: U7 carry {sharpe_net(carry[u], u)} | basis {sharpe_net(basisU[u], u)}')
for m in ('zmean', 'mean', 'median', 'worst'):
    sc = fx.exposure_scalar(fx.basis_stress_index(basisU, method=m).reindex(xret.index), lookback=756, q=0.80, low_mult=0.5)
    wc = w['all_carry'].mul(sc.reindex(w['all_carry'].index, method='ffill').fillna(1.0), axis=0)
    nn = (fx.portfolio_returns(wc, xret) - fx.roundtrip_cost(wc, hs_out, hs_pts)).rename('x').loc[WD3]
    print(f'conditioner method={m:6s}: ALL Sharpe {round(float(fx.summary_stats(nn.to_frame()).loc["x","sharpe"]), 3)}')

wrote: basis_carry_comparison.csv, basis_carry_spanning.csv, basis_track_correlation.csv


3M tenor : U7 carry -0.317 | 3M basis-sort 0.07


drop ['CNH']: U7 carry -0.28 | basis -0.028


drop ['TRY']: U7 carry 0.272 | basis -0.008
conditioner method=zmean : ALL Sharpe 0.308
conditioner method=mean  : ALL Sharpe 0.274
conditioner method=median: ALL Sharpe 0.321
conditioner method=worst : ALL Sharpe 0.326


## 7. Verdict — does the cross-currency basis beat vanilla carry net of costs?

**REJECT — another honest null, decisively.** The basis is only measurable where onshore fixings exist,
which restricts the tradable universe to **7 EM names** {CNH, HUF, ILS, INR, PLN, THB, TRY} — and on
that universe the matched vanilla carry book is itself **negative** (net Sharpe **−0.32** over
2007–2024), far below both published bars (0.466, 0.457). Every basis variant is also negative:

| variant | net Sharpe | α vs carry (ann) | t | verdict |
|---|---|---|---|---|
| `basis` (basis-sorted) | **−0.13** | −0.9% | −0.3 | reject |
| `onshore` (pure-rate carry) | **−0.41** | −1.5% | −1.2 | reject |
| `tilthi` (toward rich basis) | **−0.17** | +0.9% | +0.5 | reject |
| `tiltlo` (toward dollar-shortage) | **−0.69** | −5.7% | −2.8 | reject (sig. worse) |
| `clean` (carry ⟂ basis) | **−0.20** | +0.7% | +0.5 | reject |
| **conditioner** on ALL-27 | **0.31** (vs 0.28 uncond.) | +0.4% | +0.8 | reject (< 0.466) |

- **The basis-tradable universe is a carry graveyard.** These are exactly the restricted/convertibility-
  constrained EM where you *can* observe a dollar-funding basis — and where carry is a minefield. The
  negative anchor is driven by the TRY lira collapse (drop TRY → carry flips to +0.27, still far below
  the bars; drop CNH → −0.28); it is a universe-structural result, not a single quirk.
- **Cross-sectionally the basis adds nothing that clears the bar.** Both tilts and the orthogonalized
  "clean" carry land near the (already-negative) anchor with insignificant alpha; only `tiltlo`
  (chasing the most dollar-short names, Du's "shortage pays") is *significantly worse* (t −2.8).
- **Spanning is a two-way non-result on a weak universe.** Neither `BASIS~CARRY` (α +1.6%/yr, t +0.7)
  nor `CARRY~BASIS` (α −0.8%/yr, t −0.3) is significant — carry is too weak here to span or be spanned.
  The onshore-rate carry is a near-clone of the forward carry (β 0.89, R² 0.71): the basis is a small,
  non-priced increment, exactly as the `carry = onshore + basis` identity implies.
- **As a conditioner the basis times nothing tradable.** The zmean funding-stress index does spike at
  Lehman-2008 and the COVID-2020 dollar squeeze (it measures what DTV/Avdjiev say it does), but halving
  ALL-27 exposure in its top quintile lifts the book only 0.28→0.31 with insignificant alpha (t +0.8)
  and stays far under 0.466 — the same verdict Stage-6 regime-timing reached with VIX/vol/EMBI. Robust
  across the 3M tenor and all four stress methods.

**Bottom line.** The cross-currency basis is economically real — the audience-relevant Global-Funding
signal — but on the only universe where it is measurable here it is **not** a tradable carry edge net
of costs. D3 is a clean null, a valid deliverable that reinforces the through-line: every embellishment,
standard *and* novel, fails to beat the simple vol-targeted EM-carry book.

In [8]:
# ===== 7. Verdict block — net decision metrics + the two bars =====
show = ['sharpe', 'info_ratio', 'max_drawdown', 'CVaR_99', 'skew', 'ann_cost_drag', 't_alpha_vs_carry']
rows = ['U7_carry_net', 'U7_basis_net', 'U7_onshore_net', 'U7_tilthi_net',
        'U7_tiltlo_net', 'U7_clean_net', 'ALL_carry_net', 'ALL_basiscond_net']
print('=== D3 basis battery — NET (matched 7-name EM universe, 2007-05..2024-09) ===')
print(summary.loc[rows, show].round(3))
print(f"\nBars to beat: simple vol-targeted ALL 0.466 | per-ccy RR 0.457 "
      f"| matched U7 carry {float(summary.loc['U7_carry_net', 'sharpe']):.3f} "
      f"| ALL-27 carry (2007-24 subsample) {float(summary.loc['ALL_carry_net', 'sharpe']):.3f}")
print('\n=== basis vs carry spanning (gross factor returns, NW 5 lags) ===')
print(span.round(4))
print('\nVerdict: REJECT — the basis-tradable universe is a carry graveyard (matched carry −0.32); '
      'no basis variant or conditioner beats the bars; carry and basis fail to span each other.')

=== D3 basis battery — NET (matched 7-name EM universe, 2007-05..2024-09) ===
                     sharpe info_ratio max_drawdown   CVaR_99      skew  \
U7_carry_net      -0.317246  -0.357128    -0.716549  0.029722 -0.291386   
U7_basis_net      -0.125632   -0.14086    -0.535594   0.02959 -0.106062   
U7_onshore_net     -0.40838  -0.463473    -0.744791  0.028567 -0.347961   
U7_tilthi_net     -0.171261  -0.205843    -0.666208  0.033007 -0.727703   
U7_tiltlo_net     -0.687905   -0.77148     -0.82983  0.029781 -0.612964   
U7_clean_net      -0.200174  -0.246795    -0.620764   0.02699 -0.589196   
ALL_carry_net      0.279244   0.210128    -0.293185  0.029292 -0.620655   
ALL_basiscond_net  0.307949   0.227253    -0.277317  0.028376  -0.66631   

                   ann_cost_drag  t_alpha_vs_carry  
U7_carry_net               0.018               NaN  
U7_basis_net               0.034            -0.322  
U7_onshore_net             0.013            -1.230  
U7_tilthi_net              0.028  